# Midterm Project: Stroke Prediction 🧠

## 1. Problem Description
A stroke is a medical emergency that occurs when blood flow to the brain is interrupted. Early identification of individuals at high risk for a stroke allows healthcare providers to implement preventive measures, ultimately saving lives and reducing medical costs.

The goal of this project is to build a **Binary Classification** machine learning model that predicts whether a patient is likely to suffer a stroke (`stroke = 1`) or not (`stroke = 0`) based on their demographic and health records (e.g., age, average glucose level, BMI, hypertension, and smoking status).

## 2. Dataset
We are using a modified version of the **Healthcare Dataset - Stroke Data** from Kaggle.
* **Target Variable:** `stroke` (1 = stroke, 0 = no stroke).
* **Problem Type:** Binary Classification.

Let's begin by importing the necessary libraries, loading the data, and performing initial data cleaning (simulating missing values and outliers).

In [68]:
import pandas as pd
import numpy as np



# Read data

In [69]:
df=pd.read_csv("healthcare-dataset-stroke-data_manipulated.csv")
df.head()


,id,gender,age,hypertension,heart disease,ever married,work type,Residence type,avg glucose level,bmi,smoking status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


#  Clean column names (convert them to lowercase letters and replace spaces with underscores

In [70]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [71]:
df.columns = df.columns.str.replace('__', '_')

In [72]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


#  Clean text columns (categorical) to ensure there are no conflicting uppercase or lowercase letters

In [73]:
categorical_columns=list(df.select_dtypes(include=['object', 'str']).columns)
for col in categorical_columns:
    df[col]=df[col].astype(str).str.lower().str.replace(" ","_")
    

#  Dealing with missing values (Missing Values)

In [74]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col]=df[col].fillna(df[col].medialn())
        else:
            df[col]=df[col].fillna(df[col].mode()[0])

In [75]:
print(df.isnull().sum())    
print(df.duplicated().sum())

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64
0


#  Handling outliers (Outliers) in a BMI column

In [76]:
if 'bmi' in df.columns:
    df['bmi']=df['bmi'].clip(upper=60)

cols_to_describe = ['age', 'avg_glucose_level', 'bmi']
print("Dataset Shape:", df.shape)
print("\nNumerical Summary after handling outliers:\n", df[cols_to_describe].describe().round(2))
#print("\nMissing values after cleaning:\n", df.isnull().sum().sum())
#print("\nNumerical Summary after handling outliers:\n", df[['age', 'avg__glucose__level', 'bmi']].describe())

Dataset Shape: (4909, 12)

Numerical Summary after handling outliers:
            age  avg_glucose_level      bmi
count  4909.00            4909.00  4909.00
mean     42.87             105.31    28.87
std      22.56              44.42     7.71
min       0.08              55.12    10.30
25%      25.00              77.07    23.50
50%      44.00              91.68    28.10
75%      60.00             113.57    33.10
max      82.00             271.74    60.00


# Split Data (60% Train, 20% Val, 20% Test)

In [77]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score



In [78]:
df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train = df_train['stroke'].values
y_val = df_val['stroke'].values
y_test = df_test['stroke'].values


del df_train['stroke']
del df_val['stroke']
del df_test['stroke']


print("Train shape:", df_train.shape)

print("--- Mutual Information (Categorical vs Stroke) ---")

Train shape: (2945, 11)
--- Mutual Information (Categorical vs Stroke) ---


In [79]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

In [80]:
mi_scores = df_train[categorical_columns].apply(calculate_mi).sort_values(ascending=False)
print(mi_scores.round(4))

ever_married      0.0239
work_type         0.0181
smoking_status    0.0124
gender            0.0018
residence_type    0.0000
dtype: float64


In [81]:
print("\n--- Correlation (Numerical vs Stroke) ---")



--- Correlation (Numerical vs Stroke) ---


# Numerical Features Correlation

In [82]:
num_cols = list(df.select_dtypes(include=['number']).columns)
num_features = [col for col in num_cols if col not in ['id', 'stroke']]
df_train_temp = df_train.copy()
df_train_temp['stroke'] = y_train

corr = df_train_temp[num_features + ['stroke']].corr()['stroke'].drop('stroke').sort_values(ascending=False)
print(corr.round(4))

avg_glucose_level    0.7142
hypertension         0.4481
age                  0.3624
bmi                  0.1965
heart_disease        0.1757
Name: stroke, dtype: float64


In [86]:
from sklearn.feature_extraction import DictVectorizer

# تحديد الأعمدة الرقمية والفئوية المستخدمة للتدريب
numerical = ['creditscore', 'age', 'tenure', 'balance', 'numofproducts', 'estimatedsalary', 'point_earned']
categorical = ['geography', 'gender', 'hascrcard', 'isactivemember', 'complain', 'satisfaction_score', 'card_type']
features = categorical + numerical

# تحويل البيانات إلى قواميس ثم تشفيرها
dv = DictVectorizer(sparse=False)
train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

print("تم تجهيز مصفوفات التدريب بنجاح:", X_train.shape, X_val.shape)

KeyError: "['geography', 'hascrcard', 'isactivemember', 'complain', 'satisfaction_score', 'card_type', 'creditscore', 'tenure', 'balance', 'numofproducts', 'estimatedsalary', 'point_earned'] not in index"

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [84]:
model_lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

# 2. التنبؤ بالاحتمالات على بيانات التحقق (Validation Set)
y_pred_proba = model_lr.predict_proba(X_val)[:, 1]

# 3. حساب مقياس التقييم ROC AUC
auc_lr = roc_auc_score(y_val, y_pred_proba)
print(f"Logistic Regression Validation ROC AUC: {auc_lr:.4f}")

NameError: name 'X_train' is not defined